# 잎 사진 CNN: baseline 비교와 개선 모델 학습

**런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서부터 실행하세요.
이 노트북 안에 model.py / train.py / dataset.py가 포함되어 있으므로 파일을 따로 업로드하거나
개인 GitHub에 push할 필요가 없습니다. 공식 저장소에서 데이터와 baseline을 자동으로 받습니다.

Residual CNN + BatchNorm + SE attention + 두 출력 헤드의 Dropout,
완만한 데이터 증강, AdamW weight decay, warmup/cosine 스케줄,
train 라벨에만 근거한 클래스 가중치와 raw/EMA 검증 비교를 사용합니다.
사전학습 가중치는 사용하지 않습니다. 높은 점수는 전체 학습 후 검증으로 확인해야 합니다.

Google Drive 연결은 처음에 사용자 인증이 필요합니다. 학습은 Colab GPU에서 실행하며,
가중치·코드·로그는 Drive에 저장합니다. 중단되면 같은 설정과 RUN_NAME으로 재개할 수 있습니다.
GPU는 T4로 시작할 수 있으며 전체 실행 시간은 첫 epoch 측정값을 참고하세요.

In [ ]:
import torch
import torchvision
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), 'Colab 런타임을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 실행 설정
처음에는 아래 기본값으로 시작하세요. 비교 실험은 RUN_NAME을 새 이름으로 바꾸고 진행합니다.
재개할 때는 학습 설정과 총 EPOCHS를 그대로 유지해야 합니다.
RUN_BASELINE=False는 공식 baseline 비교를 생략합니다. 검증 성능을 비교하려면 True를 사용하세요.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
RUN_NAME = 'improved_seed42'
SEED = 42
EPOCHS = 40
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 3e-4
CLASS_WEIGHT_POWER = 0.35  # 0이면 클래스 가중치 해제
RUN_BASELINE = True
BASELINE_EPOCHS = 10

RUN_ROOT = Path('/content/drive/MyDrive/DeepLearning2026-2/runs')
RUN_DIR = RUN_ROOT / RUN_NAME
WORK = Path('/content/leaf-cnn/project1')
WORK.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

## 공식 코드·데이터 자동 준비
재현성을 위해 확인한 공식 저장소 커밋을 고정합니다. 이미 있는 checkout을 자동으로 갱신하지 않습니다.

In [ ]:
import subprocess
import sys

OFFICIAL_COMMIT = 'c0c7737fa2355cf57bf78792b376bf770aa950dd'
OFFICIAL = Path('/content/Class2026_DL-' + OFFICIAL_COMMIT[:8])
if not OFFICIAL.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', 'main',
                    'https://github.com/maygodwithu/Class2026_DL.git', str(OFFICIAL)], check=True)
    head = subprocess.check_output(['git', '-C', str(OFFICIAL), 'rev-parse', 'HEAD'], text=True).strip()
    if head != OFFICIAL_COMMIT:
        subprocess.run(['git', '-C', str(OFFICIAL), 'fetch', '--depth', '1', 'origin', OFFICIAL_COMMIT], check=True)
        subprocess.run(['git', '-C', str(OFFICIAL), 'checkout', '--detach', OFFICIAL_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(OFFICIAL), 'rev-parse', 'HEAD'], text=True).strip()
assert head == OFFICIAL_COMMIT, '공식 저장소 커밋이 다릅니다. 기존 checkout을 확인하세요.'
assert not subprocess.check_output(['git', '-C', str(OFFICIAL), 'status', '--porcelain'], text=True).strip(), '공식 코드에 변경 사항이 있습니다.'
OFFICIAL_PROJECT = OFFICIAL / 'project1'
DATA_DIR = OFFICIAL_PROJECT / 'student_data_128'
for name in ['train.csv', 'valid.csv', 'class_mapping.json']:
    assert (DATA_DIR / name).is_file(), name
print('데이터 경로:', DATA_DIR)

## 개선 학습 파일 자동 생성
아래 셀은 노트북에 포함된 소스를 Colab 작업 폴더에 씁니다. 수동 파일 업로드는 필요 없습니다.
이미 수정한 파일이 있으면 덮어쓰지 않고 중단합니다. 파일을 편집한 뒤에는 기존 작업 폴더에서 실행하세요.

In [ ]:
SOURCES = {'model.py': '"""A small residual CNN trained from scratch; only torch is required at '
             'inference."""\n'
             'import torch\n'
             'from torch import nn\n'
             '\n'
             '\n'
             'class ChannelAttention(nn.Module):\n'
             '    def __init__(self, channels):\n'
             '        super().__init__()\n'
             '        hidden = max(8, channels // 8)\n'
             '        self.layers = nn.Sequential(\n'
             '            nn.AdaptiveAvgPool2d(1),\n'
             '            nn.Conv2d(channels, hidden, 1),\n'
             '            nn.ReLU(inplace=True),\n'
             '            nn.Conv2d(hidden, channels, 1),\n'
             '            nn.Sigmoid(),\n'
             '        )\n'
             '\n'
             '    def forward(self, x):\n'
             '        return x * self.layers(x)\n'
             '\n'
             '\n'
             'class ResidualBlock(nn.Module):\n'
             '    def __init__(self, in_channels, out_channels, stride=1):\n'
             '        super().__init__()\n'
             '        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)\n'
             '        self.bn1 = nn.BatchNorm2d(out_channels)\n'
             '        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, '
             'bias=False)\n'
             '        self.bn2 = nn.BatchNorm2d(out_channels)\n'
             '        self.attention = ChannelAttention(out_channels)\n'
             '        self.relu = nn.ReLU(inplace=True)\n'
             '        self.shortcut = nn.Identity()\n'
             '        if stride != 1 or in_channels != out_channels:\n'
             '            self.shortcut = nn.Sequential(\n'
             '                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),\n'
             '                nn.BatchNorm2d(out_channels),\n'
             '            )\n'
             '\n'
             '    def forward(self, x):\n'
             '        residual = self.shortcut(x)\n'
             '        x = self.relu(self.bn1(self.conv1(x)))\n'
             '        x = self.attention(self.bn2(self.conv2(x)))\n'
             '        return self.relu(x + residual)\n'
             '\n'
             '\n'
             'class MultiTaskCNN(nn.Module):\n'
             '    def __init__(self, num_crops=3, num_health=2):\n'
             '        super().__init__()\n'
             '        self.stem = nn.Sequential(\n'
             '            nn.Conv2d(3, 32, 3, padding=1, bias=False),\n'
             '            nn.BatchNorm2d(32),\n'
             '            nn.ReLU(inplace=True),\n'
             '        )\n'
             '        self.features = nn.Sequential(\n'
             '            ResidualBlock(32, 32, 2),\n'
             '            ResidualBlock(32, 64, 2),\n'
             '            ResidualBlock(64, 128, 2),\n'
             '            ResidualBlock(128, 128),\n'
             '            ResidualBlock(128, 192, 2),\n'
             '        )\n'
             '        self.avg_pool = nn.AdaptiveAvgPool2d(1)\n'
             '        self.max_pool = nn.AdaptiveMaxPool2d(1)\n'
             '        self.crop_features = self._head_features()\n'
             '        self.health_features = self._head_features()\n'
             '        self.crop_head = nn.Linear(128, num_crops)\n'
             '        self.health_head = nn.Linear(128, num_health)\n'
             '        self._initialize()\n'
             '\n'
             '    @staticmethod\n'
             '    def _head_features():\n'
             '        return nn.Sequential(\n'
             '            nn.Linear(384, 128),\n'
             '            nn.LayerNorm(128),\n'
             '            nn.ReLU(inplace=True),\n'
             '            nn.Dropout(0.2),\n'
             '        )\n'
             '\n'
             '    def _initialize(self):\n'
             '        for module in self.modules():\n'
             '            if isinstance(module, nn.Conv2d):\n'
             '                nn.init.kaiming_normal_(module.weight, mode="fan_out", '
             'nonlinearity="relu")\n'
             '                if module.bias is not None:\n'
             '                    nn.init.zeros_(module.bias)\n'
             '            elif isinstance(module, (nn.BatchNorm2d, nn.LayerNorm)):\n'
             '                nn.init.ones_(module.weight)\n'
             '                nn.init.zeros_(module.bias)\n'
             '            elif isinstance(module, nn.Linear):\n'
             '                nn.init.normal_(module.weight, std=0.01)\n'
             '                nn.init.zeros_(module.bias)\n'
             '        for module in self.modules():\n'
             '            if isinstance(module, ResidualBlock):\n'
             '                nn.init.zeros_(module.bn2.weight)\n'
             '\n'
             '    def forward(self, x):\n'
             '        x = self.features(self.stem(x))\n'
             '        features = torch.cat((self.avg_pool(x).flatten(1), '
             'self.max_pool(x).flatten(1)), dim=1)\n'
             '        return (\n'
             '            self.crop_head(self.crop_features(features)),\n'
             '            self.health_head(self.health_features(features)),\n'
             '        )\n',
 'dataset.py': '"""Official CSV format, with training-only augmentation and fixed grader '
               'transforms."""\n'
               'import csv\n'
               'import json\n'
               'from pathlib import Path\n'
               '\n'
               'from PIL import Image\n'
               'from torch.utils.data import Dataset\n'
               'from torchvision import transforms\n'
               'from torchvision.transforms import InterpolationMode\n'
               '\n'
               '\n'
               'def load_class_names(data_dir):\n'
               '    with (Path(data_dir) / "class_mapping.json").open(encoding="utf-8") as '
               'handle:\n'
               '        mapping = json.load(handle)\n'
               '    return tuple(\n'
               '        [mapping[key][str(i)] for i in range(len(mapping[key]))]\n'
               '        for key in ("crop_label", "health_label")\n'
               '    )\n'
               '\n'
               '\n'
               'class PlantDataset(Dataset):\n'
               '    def __init__(self, data_dir, split, transform=None):\n'
               '        self.img_dir = Path(data_dir) / split / "images"\n'
               '        self.transform = transform\n'
               '        with (Path(data_dir) / f"{split}.csv").open(encoding="utf-8-sig", '
               'newline="") as handle:\n'
               '            self.rows = list(csv.DictReader(handle))\n'
               '        if not self.rows:\n'
               '            raise ValueError(f"Empty split: {split}")\n'
               '        for row in self.rows:\n'
               '            if not 0 <= int(row["crop_label"]) < 3 or not 0 <= '
               'int(row["health_label"]) < 2:\n'
               '                raise ValueError(f"Invalid labels: {row}")\n'
               '            if not (self.img_dir / row["id"]).is_file():\n'
               '                raise FileNotFoundError(self.img_dir / row["id"])\n'
               '\n'
               '    def __len__(self):\n'
               '        return len(self.rows)\n'
               '\n'
               '    def __getitem__(self, index):\n'
               '        row = self.rows[index]\n'
               '        with Image.open(self.img_dir / row["id"]) as handle:\n'
               '            image = handle.convert("RGB")\n'
               '        if self.transform is not None:\n'
               '            image = self.transform(image)\n'
               '        return image, int(row["crop_label"]), int(row["health_label"])\n'
               '\n'
               '\n'
               'def get_transforms(train, augment=True):\n'
               '    normalize = transforms.Normalize([0.5] * 3, [0.5] * 3)\n'
               '    if train and augment:\n'
               '        return transforms.Compose([\n'
               '            transforms.RandomResizedCrop(\n'
               '                128, scale=(0.8, 1.0), ratio=(0.9, 1.1),\n'
               '                interpolation=InterpolationMode.BILINEAR,\n'
               '            ),\n'
               '            transforms.RandomHorizontalFlip(),\n'
               '            transforms.RandomVerticalFlip(),\n'
               '            transforms.RandomRotation(25, '
               'interpolation=InterpolationMode.BILINEAR),\n'
               '            transforms.ColorJitter(brightness=0.15, contrast=0.15, '
               'saturation=0.12, hue=0.02),\n'
               '            transforms.ToTensor(),\n'
               '            normalize,\n'
               '        ])\n'
               '    # Do not change this: the official grader uses this exact preprocessing.\n'
               '    return transforms.Compose([transforms.ToTensor(), normalize])\n',
 'train.py': '"""Train from scratch, select by mean Macro-F1, and retain matching submission '
             'code."""\n'
             'import argparse\n'
             'import copy\n'
             'import csv\n'
             'import hashlib\n'
             'import json\n'
             'import math\n'
             'import os\n'
             'from pathlib import Path\n'
             'import random\n'
             'import shutil\n'
             'import time\n'
             '\n'
             'import torch\n'
             'from torch import nn\n'
             'from torch.utils.data import DataLoader, Subset\n'
             '\n'
             'from dataset import PlantDataset, get_transforms, load_class_names\n'
             'from model import MultiTaskCNN\n'
             '\n'
             '\n'
             'def atomic_save(value, path):\n'
             '    path = Path(path)\n'
             '    temporary = path.with_name(path.name + ".tmp")\n'
             '    torch.save(value, temporary)\n'
             '    os.replace(temporary, path)\n'
             '\n'
             '\n'
             'def write_json(value, path):\n'
             '    path = Path(path)\n'
             '    temporary = path.with_name(path.name + ".tmp")\n'
             '    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), '
             'encoding="utf-8")\n'
             '    os.replace(temporary, path)\n'
             '\n'
             '\n'
             'def seed_worker(worker_id):\n'
             '    random.seed(torch.initial_seed() % (2 ** 32))\n'
             '\n'
             '\n'
             'def macro_f1(conf):\n'
             '    true_positive = conf.diag().float()\n'
             '    precision = true_positive / conf.sum(0).clamp(min=1)\n'
             '    recall = true_positive / conf.sum(1).clamp(min=1)\n'
             '    f1 = 2 * precision * recall / (precision + recall).clamp(min=1e-8)\n'
             '    return f1.mean().item(), f1.tolist()\n'
             '\n'
             '\n'
             '@torch.no_grad()\n'
             'def evaluate(model, loader, device):\n'
             '    model.eval()\n'
             '    matrices = [torch.zeros(3, 3, dtype=torch.long), torch.zeros(2, 2, '
             'dtype=torch.long)]\n'
             '    for images, crops, healths in loader:\n'
             '        outputs = model(images.to(device, non_blocking=True))\n'
             '        for matrix, labels, logits, classes in zip(matrices, (crops, healths), '
             'outputs, (3, 2)):\n'
             '            if logits.shape != (len(images), classes) or not '
             'torch.isfinite(logits).all():\n'
             '                raise ValueError("Invalid output shape or non-finite validation '
             'logits")\n'
             '            encoded = labels * classes + logits.argmax(1).cpu()\n'
             '            matrix += torch.bincount(encoded, minlength=classes * '
             'classes).reshape(classes, classes)\n'
             '    result = {}\n'
             '    for key, matrix in zip(("crop", "health"), matrices):\n'
             '        f1, per_class = macro_f1(matrix)\n'
             '        result.update({\n'
             '            key + "_f1": f1, key + "_f1_per_class": per_class,\n'
             '            key + "_accuracy": (matrix.diag().sum() / matrix.sum()).item(),\n'
             '            key + "_confusion": matrix.tolist(),\n'
             '        })\n'
             '    result["score"] = (result["crop_f1"] + result["health_f1"]) / 2\n'
             '    return result\n'
             '\n'
             '\n'
             'def print_report(result, names):\n'
             '    for key, labels in zip(("crop", "health"), names):\n'
             '        print(f"{key}: Macro-F1 {result[key + \'_f1\']:.4f}")\n'
             '        for label, value in zip(labels, result[key + "_f1_per_class"]):\n'
             '            print(f"  {label}: {value:.4f}")\n'
             '        print("  confusion (true rows, predicted columns):", result[key + '
             '"_confusion"])\n'
             '    print(f"Mean Macro-F1: {result[\'score\']:.4f}")\n'
             '\n'
             '\n'
             'def class_weights(rows, key, classes, power, device):\n'
             '    counts = torch.bincount(torch.tensor([int(row[key]) for row in rows]), '
             'minlength=classes).float()\n'
             '    weights = counts.clamp(min=1).pow(-power)\n'
             '    weights /= weights.mean()\n'
             '    print(f"{key}: counts {counts.int().tolist()}, loss weights '
             '{weights.tolist()}")\n'
             '    return weights.to(device)\n'
             '\n'
             '\n'
             'class ModelEMA:\n'
             '    def __init__(self, model, decay):\n'
             '        self.model = copy.deepcopy(model).eval().requires_grad_(False)\n'
             '        self.decay = decay\n'
             '        self.steps = 0\n'
             '\n'
             '    @torch.no_grad()\n'
             '    def update(self, model):\n'
             '        self.steps += 1\n'
             '        decay = min(self.decay, (1 + self.steps) / (10 + self.steps))\n'
             '        for averaged, current in zip(self.model.parameters(), model.parameters()):\n'
             '            averaged.lerp_(current.detach(), 1 - decay)\n'
             '        # Keep BN statistics current; validate EMA separately before selecting it.\n'
             '        for averaged, current in zip(self.model.buffers(), model.buffers()):\n'
             '            averaged.copy_(current)\n'
             '\n'
             '\n'
             'def parse_args():\n'
             '    parser = argparse.ArgumentParser(description=__doc__)\n'
             '    parser.add_argument("--data_dir", default="student_data_128")\n'
             '    parser.add_argument("--run_dir", default="runs/improved_seed42")\n'
             '    parser.add_argument("--epochs", type=int, default=40)\n'
             '    parser.add_argument("--batch_size", type=int, default=64)\n'
             '    parser.add_argument("--lr", type=float, default=1e-3)\n'
             '    parser.add_argument("--min_lr", type=float, default=1e-5)\n'
             '    parser.add_argument("--warmup_epochs", type=int, default=3)\n'
             '    parser.add_argument("--weight_decay", type=float, default=3e-4)\n'
             '    parser.add_argument("--class_weight_power", type=float, default=0.35,\n'
             '                        help="0 disables class weighting; only train labels '
             'determine weights")\n'
             '    parser.add_argument("--label_smoothing", type=float, default=0.03)\n'
             '    parser.add_argument("--ema_decay", type=float, default=0.99, help="0 disables '
             'EMA")\n'
             '    parser.add_argument("--seed", type=int, default=42)\n'
             '    parser.add_argument("--num_workers", type=int, default=2)\n'
             '    parser.add_argument("--num_threads", type=int, default=2)\n'
             '    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), '
             'default="auto")\n'
             '    parser.add_argument("--no_amp", action="store_true")\n'
             '    parser.add_argument("--no_augment", action="store_true")\n'
             '    parser.add_argument("--resume", action="store_true")\n'
             '    parser.add_argument("--eval_only", action="store_true")\n'
             '    parser.add_argument("--train_limit", type=int, default=0, help="Smoke checks '
             'only, 0 uses all data")\n'
             '    parser.add_argument("--valid_limit", type=int, default=0, help="Smoke checks '
             'only, 0 uses all data")\n'
             '    args = parser.parse_args()\n'
             '    if args.epochs < 1 or args.batch_size < 2 or args.lr <= 0 or not 0 <= '
             'args.min_lr <= args.lr:\n'
             '        parser.error("Require epochs >= 1, batch_size >= 2 and 0 <= min_lr <= lr '
             'with lr > 0")\n'
             '    if args.warmup_epochs < 0 or args.weight_decay < 0 or args.class_weight_power < '
             '0:\n'
             '        parser.error("Warmup, decay and class_weight_power cannot be negative")\n'
             '    if not 0 <= args.ema_decay < 1 or not 0 <= args.label_smoothing < 1:\n'
             '        parser.error("EMA decay and label smoothing must be in [0, 1)")\n'
             '    if min(args.num_workers, args.train_limit, args.valid_limit) < 0 or '
             'args.num_threads < 1:\n'
             '        parser.error("Invalid worker, thread or subset counts")\n'
             '    return args\n'
             '\n'
             '\n'
             'def main():\n'
             '    args = parse_args()\n'
             '    random.seed(args.seed)\n'
             '    torch.manual_seed(args.seed)\n'
             '    torch.set_num_threads(args.num_threads)\n'
             '    if torch.cuda.is_available():\n'
             '        torch.cuda.manual_seed_all(args.seed)\n'
             '    torch.backends.cudnn.benchmark = False\n'
             '    torch.backends.cudnn.deterministic = True\n'
             '    device = torch.device("cuda" if args.device == "auto" and '
             'torch.cuda.is_available()\n'
             '                          else "cpu" if args.device == "auto" else args.device)\n'
             '    if device.type == "cuda" and not torch.cuda.is_available():\n'
             '        raise RuntimeError("CUDA unavailable; enable a Colab GPU or select --device '
             'cpu")\n'
             '    run_dir = Path(args.run_dir).resolve()\n'
             '    source_dir = Path(__file__).resolve().parent\n'
             '    run_dir.mkdir(parents=True, exist_ok=True)\n'
             '    best_path, last_path = run_dir / "best_model.pt", run_dir / '
             '"last_checkpoint.pt"\n'
             '    if not args.resume and not args.eval_only and any(\n'
             '        (run_dir / name).exists() for name in ("config.json", "best_model.pt", '
             '"last_checkpoint.pt")\n'
             '    ):\n'
             '        raise FileExistsError("Run exists: choose a new --run_dir or pass '
             '--resume")\n'
             '    hashes = {name: hashlib.sha256((source_dir / name).read_bytes()).hexdigest()\n'
             '              for name in ("model.py", "dataset.py", "train.py")}\n'
             '    if args.resume or args.eval_only:\n'
             '        existing = json.loads((run_dir / "config.json").read_text())\n'
             '        if existing["source_sha256"] != hashes:\n'
             '            raise ValueError("Source changed since this run; use the saved run '
             'sources or a new run")\n'
             '    else:\n'
             '        for name in hashes:\n'
             '            shutil.copy2(source_dir / name, run_dir / name)\n'
             '        write_json({"args": vars(args), "source_sha256": hashes,\n'
             '                    "torch": torch.__version__, "device": str(device),\n'
             '                    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" '
             'else None},\n'
             '                   run_dir / "config.json")\n'
             '    print(f"Device: {device}; run: {run_dir}", flush=True)\n'
             '    if args.train_limit or args.valid_limit:\n'
             '        print("SMOKE RUN: subset metrics are not full validation or submission '
             'performance.")\n'
             '    names = load_class_names(args.data_dir)\n'
             '    if tuple(map(len, names)) != (3, 2):\n'
             '        raise ValueError("Expected 3 crop classes and 2 health classes")\n'
             '    valid_set = PlantDataset(args.data_dir, "valid", get_transforms(False))\n'
             '    if args.valid_limit:\n'
             '        valid_set = Subset(valid_set, range(min(args.valid_limit, len(valid_set))))\n'
             '    valid_loader = DataLoader(valid_set, batch_size=args.batch_size, '
             'num_workers=args.num_workers,\n'
             '                              pin_memory=device.type == "cuda")\n'
             '    model = MultiTaskCNN().to(device)\n'
             '    count = sum(p.numel() for p in model.parameters()) + sum(\n'
             '        b.numel() for b in model.buffers() if b.is_floating_point())\n'
             '    print(f"Parameters + floating buffers: {count:,} / 2,000,000")\n'
             '    if count > 2_000_000:\n'
             '        raise ValueError("Model exceeds submission size limit")\n'
             '    if args.eval_only:\n'
             '        model.load_state_dict(torch.load(best_path, map_location=device, '
             'weights_only=True))\n'
             '        result = evaluate(model, valid_loader, device)\n'
             '        print_report(result, names)\n'
             '        write_json(result, run_dir / "evaluation.json")\n'
             '        return\n'
             '\n'
             '    train_set = PlantDataset(args.data_dir, "train", get_transforms(True, not '
             'args.no_augment))\n'
             '    rows = train_set.rows\n'
             '    if args.train_limit:\n'
             '        indices = torch.randperm(len(train_set), '
             'generator=torch.Generator().manual_seed(args.seed))[\n'
             '            :args.train_limit].tolist()\n'
             '        rows = [rows[i] for i in indices]\n'
             '        train_set = Subset(train_set, indices)\n'
             '    if len(train_set) < 2:\n'
             '        raise ValueError("Need at least two training images")\n'
             '    generator = torch.Generator().manual_seed(args.seed)\n'
             '    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True,\n'
             '                              num_workers=args.num_workers, pin_memory=device.type '
             '== "cuda",\n'
             '                              worker_init_fn=seed_worker, generator=generator)\n'
             '    criteria = [nn.CrossEntropyLoss(\n'
             '        weight=class_weights(rows, key, classes, args.class_weight_power, device),\n'
             '        label_smoothing=args.label_smoothing,\n'
             '    ) for key, classes in (("crop_label", 3), ("health_label", 2))]\n'
             '    decay, no_decay = [], []\n'
             '    for parameter in model.parameters():\n'
             '        (decay if parameter.ndim > 1 else no_decay).append(parameter)\n'
             '    optimizer = torch.optim.AdamW([\n'
             '        {"params": decay, "weight_decay": args.weight_decay},\n'
             '        {"params": no_decay, "weight_decay": 0.0},\n'
             '    ], lr=args.lr)\n'
             '    warmup = min(args.warmup_epochs, max(0, args.epochs - 1))\n'
             '\n'
             '    def lr_factor(epoch):\n'
             '        if epoch < warmup:\n'
             '            return (epoch + 1) / max(1, warmup)\n'
             '        progress = min(1.0, (epoch - warmup) / max(1, args.epochs - warmup - 1))\n'
             '        minimum = args.min_lr / args.lr\n'
             '        return minimum + (1 - minimum) * (1 + math.cos(math.pi * progress)) / 2\n'
             '\n'
             '    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)\n'
             '    amp = device.type == "cuda" and not args.no_amp\n'
             '    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n'
             '    ema = ModelEMA(model, args.ema_decay) if args.ema_decay else None\n'
             '    history, best_score, first_epoch = [], -1.0, 1\n'
             '    if args.resume:\n'
             '        saved = torch.load(last_path, map_location="cpu", weights_only=True)\n'
             '        ignored = {"resume", "eval_only", "run_dir", "device", "num_workers", '
             '"num_threads"}\n'
             '        if any(saved["args"][key] != value for key, value in vars(args).items() if '
             'key not in ignored):\n'
             '            raise ValueError("Resume requires the same training options, including '
             'total epochs")\n'
             '        model.load_state_dict(saved["model"])\n'
             '        optimizer.load_state_dict(saved["optimizer"])\n'
             '        scheduler.load_state_dict(saved["scheduler"])\n'
             '        scaler.load_state_dict(saved["scaler"])\n'
             '        if ema:\n'
             '            ema.model.load_state_dict(saved["ema"])\n'
             '            ema.steps = saved["ema_steps"]\n'
             '        history, best_score = saved["history"], saved["best_score"]\n'
             '        first_epoch = saved["epoch"] + 1\n'
             '        random.setstate(saved["random_state"])\n'
             '        torch.set_rng_state(saved["torch_rng"])\n'
             '        generator.set_state(saved["loader_rng"])\n'
             '        if device.type == "cuda" and saved["cuda_rng"]:\n'
             '            torch.cuda.set_rng_state_all(saved["cuda_rng"])\n'
             '        print(f"Resuming epoch {first_epoch}; best score {best_score:.4f}")\n'
             '\n'
             '    print(f"Train {len(train_set)}; valid {len(valid_set)}; AMP {amp}", flush=True)\n'
             '    for epoch in range(first_epoch, args.epochs + 1):\n'
             '        start = time.perf_counter()\n'
             '        model.train()\n'
             '        total_loss, seen, updates = 0.0, 0, 0\n'
             '        learning_rate = optimizer.param_groups[0]["lr"]\n'
             '        for images, crops, healths in train_loader:\n'
             '            images = images.to(device, non_blocking=True)\n'
             '            crops, healths = crops.to(device, non_blocking=True), healths.to(device, '
             'non_blocking=True)\n'
             '            optimizer.zero_grad(set_to_none=True)\n'
             '            with torch.amp.autocast(device_type=device.type, enabled=amp):\n'
             '                crop, health = model(images)\n'
             '                loss = criteria[0](crop, crops) + criteria[1](health, healths)\n'
             '            if not torch.isfinite(loss):\n'
             '                raise FloatingPointError("Non-finite training loss")\n'
             '            scaler.scale(loss).backward()\n'
             '            scaler.unscale_(optimizer)\n'
             '            gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), '
             'max_norm=5.0)\n'
             '            if not amp and not torch.isfinite(gradient_norm):\n'
             '                raise FloatingPointError("Non-finite gradients")\n'
             '            old_scale = scaler.get_scale()\n'
             '            scaler.step(optimizer)\n'
             '            scaler.update()\n'
             '            if scaler.get_scale() >= old_scale:\n'
             '                updates += 1\n'
             '                if ema:\n'
             '                    ema.update(model)\n'
             '            total_loss += loss.item() * len(images)\n'
             '            seen += len(images)\n'
             '        if not updates:\n'
             '            raise FloatingPointError("No optimizer updates completed")\n'
             '        raw_result = evaluate(model, valid_loader, device)\n'
             '        ema_result = evaluate(ema.model, valid_loader, device) if ema else None\n'
             '        selected, result, variant = model, raw_result, "raw"\n'
             '        if ema_result and ema_result["score"] > raw_result["score"]:\n'
             '            selected, result, variant = ema.model, ema_result, "ema"\n'
             '        record = {"epoch": epoch, "train_loss": total_loss / seen, "lr": '
             'learning_rate,\n'
             '                  "seconds": time.perf_counter() - start, "optimizer_updates": '
             'updates,\n'
             '                  "raw_score": raw_result["score"],\n'
             '                  "ema_score": ema_result["score"] if ema_result else None,\n'
             '                  "selected": variant, **result}\n'
             '        history.append(record)\n'
             '        print(f"[{epoch:02d}/{args.epochs}] loss {record[\'train_loss\']:.4f} "\n'
             '              f"crop F1 {result[\'crop_f1\']:.4f} health F1 '
             '{result[\'health_f1\']:.4f} "\n'
             '              f"score {result[\'score\']:.4f} ({variant}) lr {learning_rate:.2g} "\n'
             '              f"{record[\'seconds\']:.1f}s", flush=True)\n'
             '        if result["score"] > best_score:\n'
             '            best_score = result["score"]\n'
             '            atomic_save({key: value.detach().cpu() for key, value in '
             'selected.state_dict().items()}, best_path)\n'
             '            if best_path.stat().st_size > 10 * 1024 * 1024:\n'
             '                raise ValueError("Checkpoint exceeds 10 MiB")\n'
             '            write_json({"epoch": epoch, "variant": variant, "weights_bytes": '
             'best_path.stat().st_size,\n'
             '                        "smoke_run": bool(args.train_limit or args.valid_limit), '
             '**result},\n'
             '                       run_dir / "best_metrics.json")\n'
             '            print(f"  Saved best_model.pt ({best_path.stat().st_size / 1024 ** '
             '2:.2f} MiB)", flush=True)\n'
             '        scheduler.step()\n'
             '        write_json(history, run_dir / "history.json")\n'
             '        with (run_dir / "history.csv").open("w", newline="", encoding="utf-8") as '
             'handle:\n'
             '            fields = ["epoch", "train_loss", "lr", "seconds", "crop_f1", '
             '"health_f1", "score",\n'
             '                      "raw_score", "ema_score", "selected", "optimizer_updates"]\n'
             '            writer = csv.DictWriter(handle, fieldnames=fields, '
             'extrasaction="ignore")\n'
             '            writer.writeheader()\n'
             '            writer.writerows(history)\n'
             '        atomic_save({\n'
             '            "epoch": epoch, "args": vars(args), "model": model.state_dict(),\n'
             '            "optimizer": optimizer.state_dict(), "scheduler": '
             'scheduler.state_dict(),\n'
             '            "scaler": scaler.state_dict(), "ema": ema.model.state_dict() if ema else '
             'None,\n'
             '            "ema_steps": ema.steps if ema else 0, "history": history, "best_score": '
             'best_score,\n'
             '            "random_state": random.getstate(), "torch_rng": torch.get_rng_state(),\n'
             '            "loader_rng": generator.get_state(),\n'
             '            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else '
             '[],\n'
             '        }, last_path)\n'
             '    model.load_state_dict(torch.load(best_path, map_location=device, '
             'weights_only=True))\n'
             '    final_result = evaluate(model, valid_loader, device)\n'
             '    print_report(final_result, names)\n'
             '    write_json(final_result, run_dir / "evaluation.json")\n'
             '\n'
             '\n'
             'if __name__ == "__main__":\n'
             '    main()\n',
 'check_submission.py': '"""제출 전 점검: 채점 서버와 같은 방식으로 model.py + best_model.pt 를 검사한다.\n'
                        '\n'
                        '사용법 (project1 폴더에서)\n'
                        '  python check_submission.py\n'
                        '  python check_submission.py --model model.py --weights best_model.pt '
                        '--data_dir student_data_128\n'
                        '\n'
                        '모든 항목이 [통과] 이면 리더보드에 제출해도 채점됩니다.\n'
                        '채점 서버처럼 CPU 로만 실행합니다.\n'
                        '"""\n'
                        'import os\n'
                        'import ast\n'
                        'import sys\n'
                        'import csv\n'
                        'import argparse\n'
                        'import importlib.util\n'
                        '\n'
                        'import torch\n'
                        'from PIL import Image\n'
                        'from torchvision import transforms\n'
                        '\n'
                        '# 채점 서버와 같은 제한\n'
                        'MAX_MODEL_BYTES = 200 * 1024\n'
                        'MAX_WEIGHT_BYTES = 10 * 1024 * 1024\n'
                        'MAX_PARAMS = 2_000_000\n'
                        'ALLOWED_PACKAGES = {"torch", "torchvision"}\n'
                        'NUM_CROPS, NUM_HEALTH = 3, 2\n'
                        '\n'
                        '# 채점 서버가 쓰는 평가 전처리 (dataset.py 의 get_transforms(train=False) 와 같음)\n'
                        'EVAL_TRANSFORM = transforms.Compose([\n'
                        '    transforms.ToTensor(),\n'
                        '    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),\n'
                        '])\n'
                        '\n'
                        '\n'
                        'def ok(msg):\n'
                        '    print(f"[통과] {msg}")\n'
                        '\n'
                        '\n'
                        'def fail(msg):\n'
                        '    print(f"[실패] {msg}")\n'
                        '    print("\\n=> 위 문제를 고친 뒤 다시 실행하세요. 이대로 제출하면 채점에 실패합니다.")\n'
                        '    sys.exit(1)\n'
                        '\n'
                        '\n'
                        'def check_imports(path):\n'
                        '    """torch, torchvision, 파이썬 기본 라이브러리 외의 import 가 있는지 검사"""\n'
                        '    tree = ast.parse(open(path, encoding="utf-8").read())\n'
                        '    names = set()\n'
                        '    for node in ast.walk(tree):\n'
                        '        if isinstance(node, ast.Import):\n'
                        '            names |= {a.name.split(".")[0] for a in node.names}\n'
                        '        elif isinstance(node, ast.ImportFrom) and node.level == 0 and '
                        'node.module:\n'
                        '            names.add(node.module.split(".")[0])\n'
                        '    return sorted(n for n in names if n not in ALLOWED_PACKAGES and n not '
                        'in sys.stdlib_module_names)\n'
                        '\n'
                        '\n'
                        'def macro_f1(pairs, num_classes):\n'
                        '    f1s = []\n'
                        '    for c in range(num_classes):\n'
                        '        tp = sum(1 for t, p in pairs if t == c and p == c)\n'
                        '        fp = sum(1 for t, p in pairs if t != c and p == c)\n'
                        '        fn = sum(1 for t, p in pairs if t == c and p != c)\n'
                        '        prec = tp / (tp + fp) if tp + fp else 0.0\n'
                        '        rec = tp / (tp + fn) if tp + fn else 0.0\n'
                        '        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)\n'
                        '    return sum(f1s) / num_classes\n'
                        '\n'
                        '\n'
                        'def main():\n'
                        '    parser = argparse.ArgumentParser()\n'
                        '    parser.add_argument("--model", default="model.py")\n'
                        '    parser.add_argument("--weights", default="best_model.pt")\n'
                        '    parser.add_argument("--data_dir", default="student_data_128")\n'
                        '    args = parser.parse_args()\n'
                        '    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))\n'
                        '\n'
                        '    print("=" * 60)\n'
                        '    print(" 제출 전 점검 (채점 서버와 같은 방식, CPU)")\n'
                        '    print("=" * 60)\n'
                        '\n'
                        '    # 1. 파일\n'
                        '    for path, limit, name in [(args.model, MAX_MODEL_BYTES, "model.py"), '
                        '(args.weights, MAX_WEIGHT_BYTES, "best_model.pt")]:\n'
                        '        if not os.path.isfile(path):\n'
                        '            fail(f"{path} 파일이 없습니다")\n'
                        '        size = os.path.getsize(path)\n'
                        '        if size > limit:\n'
                        '            fail(f"{name} 파일이 너무 큽니다: {size / 1024 / 1024:.2f}MB (제한 '
                        '{limit / 1024 / 1024:.2f}MB)")\n'
                        '    ok(f"파일 크기: model.py {os.path.getsize(args.model) / 1024:.1f}KB, "\n'
                        '       f"best_model.pt {os.path.getsize(args.weights) / 1024 / '
                        '1024:.2f}MB (제한 10MB)")\n'
                        '\n'
                        '    # 2. import\n'
                        '    bad = check_imports(args.model)\n'
                        '    if bad:\n'
                        '        fail(f"채점 서버에 없는 라이브러리를 import 합니다: {\', \'.join(bad)} (torch, '
                        'torchvision 만 사용 가능)")\n'
                        '    ok("import: torch, torchvision, 파이썬 기본 라이브러리만 사용")\n'
                        '\n'
                        '    # 3. model.py 불러오기\n'
                        '    spec = importlib.util.spec_from_file_location("student_model", '
                        'args.model)\n'
                        '    module = importlib.util.module_from_spec(spec)\n'
                        '    try:\n'
                        '        spec.loader.exec_module(module)\n'
                        '    except Exception as e:\n'
                        '        fail(f"model.py 를 불러오지 못했습니다 ({type(e).__name__}: {e})")\n'
                        '    if not hasattr(module, "MultiTaskCNN"):\n'
                        '        fail("model.py 에 MultiTaskCNN 클래스가 없습니다")\n'
                        '    ok("model.py 에 MultiTaskCNN 클래스가 있음")\n'
                        '\n'
                        '    # 4. 인자 없이 모델 만들기\n'
                        '    try:\n'
                        '        model = module.MultiTaskCNN()\n'
                        '    except Exception as e:\n'
                        '        fail(f"MultiTaskCNN() 으로 모델을 만들지 못했습니다. __init__ 의 모든 인자에 기본값이 '
                        '있어야 합니다 "\n'
                        '             f"({type(e).__name__}: {e})")\n'
                        '    ok("MultiTaskCNN() 을 인자 없이 만들 수 있음")\n'
                        '\n'
                        '    # 5. 모델 크기 (학습 파라미터 + 실수형 버퍼)\n'
                        '    n_params = sum(p.numel() for p in model.parameters())\n'
                        '    n_buffers = sum(b.numel() for b in model.buffers() if '
                        'b.is_floating_point())\n'
                        '    total = n_params + n_buffers\n'
                        '    if total > MAX_PARAMS:\n'
                        '        fail(f"모델이 너무 큽니다: 파라미터 {n_params:,}개 + 버퍼 {n_buffers:,}개 = '
                        '{total:,}개 (제한 {MAX_PARAMS:,}개)")\n'
                        '    ok(f"모델 크기: 파라미터 {n_params:,}개 + 버퍼 {n_buffers:,}개 = {total:,}개 (제한 '
                        '{MAX_PARAMS:,}개)")\n'
                        '\n'
                        '    # 6. 가중치 불러오기\n'
                        '    try:\n'
                        '        state = torch.load(args.weights, map_location="cpu", '
                        'weights_only=True)\n'
                        '    except Exception as e:\n'
                        '        fail(f"best_model.pt 를 읽지 못했습니다. torch.save(model.state_dict(), '
                        '...) 로 저장했는지 확인하세요 "\n'
                        '             f"({type(e).__name__}: {e})")\n'
                        '    try:\n'
                        '        model.load_state_dict(state)\n'
                        '    except Exception as e:\n'
                        '        fail(f"best_model.pt 가 model.py 의 구조와 맞지 않습니다. 학습할 때 쓴 model.py 와 '
                        '같은지 확인하세요 "\n'
                        '             f"({type(e).__name__}: {str(e)[:300]})")\n'
                        '    model.eval()\n'
                        '    ok("best_model.pt 를 model.py 구조에 맞게 불러옴")\n'
                        '\n'
                        '    # 7. valid 전체 예측 (출력 형식 검사 + 점수)\n'
                        '    csv_path = os.path.join(args.data_dir, "valid.csv")\n'
                        '    if not os.path.isfile(csv_path):\n'
                        '        fail(f"{csv_path} 가 없습니다. --data_dir 로 데이터 폴더를 지정하세요")\n'
                        '    with open(csv_path, encoding="utf-8-sig") as f:\n'
                        '        rows = list(csv.DictReader(f))\n'
                        '    crop_pairs, health_pairs = [], []\n'
                        '    with torch.no_grad():\n'
                        '        for i in range(0, len(rows), 128):\n'
                        '            batch = rows[i:i + 128]\n'
                        '            x = '
                        'torch.stack([EVAL_TRANSFORM(Image.open(os.path.join(args.data_dir, '
                        '"valid", "images", r["id"])).convert("RGB"))\n'
                        '                             for r in batch])\n'
                        '            try:\n'
                        '                out = model(x)\n'
                        '            except Exception as e:\n'
                        '                fail(f"모델 forward 중 오류 ({type(e).__name__}: {e})")\n'
                        '            if not (isinstance(out, (tuple, list)) and len(out) == 2):\n'
                        '                fail("모델 출력은 (crop_logits, health_logits) 두 개여야 합니다")\n'
                        '            crop_out, health_out = out\n'
                        '            n = len(batch)\n'
                        '            if tuple(crop_out.shape) != (n, NUM_CROPS) or '
                        'tuple(health_out.shape) != (n, NUM_HEALTH):\n'
                        '                fail(f"출력 크기가 올바르지 않습니다: crop {tuple(crop_out.shape)} (기대 '
                        '{(n, NUM_CROPS)}), "\n'
                        '                     f"health {tuple(health_out.shape)} (기대 {(n, '
                        'NUM_HEALTH)})")\n'
                        '            if not (torch.isfinite(crop_out).all() and '
                        'torch.isfinite(health_out).all()):\n'
                        '                fail("모델 출력에 NaN 또는 inf 가 있습니다")\n'
                        '            crop_pairs += zip([int(r["crop_label"]) for r in batch], '
                        'crop_out.argmax(1).tolist())\n'
                        '            health_pairs += zip([int(r["health_label"]) for r in batch], '
                        'health_out.argmax(1).tolist())\n'
                        '    ok(f"출력 형식: (B, 3), (B, 2)  (valid {len(rows)}장 예측 완료)")\n'
                        '\n'
                        '    crop_f1 = macro_f1(crop_pairs, NUM_CROPS)\n'
                        '    health_f1 = macro_f1(health_pairs, NUM_HEALTH)\n'
                        '    print("-" * 60)\n'
                        '    print(f" valid 점수 (채점 서버와 같은 전처리)")\n'
                        '    print(f"   작물 종류 Macro-F1 : {crop_f1:.4f}")\n'
                        '    print(f"   건강 상태 Macro-F1 : {health_f1:.4f}")\n'
                        '    print(f"   평균 Macro-F1      : {(crop_f1 + health_f1) / 2:.4f}")\n'
                        '    print("=" * 60)\n'
                        '    print(" 모든 검사 통과! 리더보드에 model.py 와 best_model.pt 를 제출하세요.")\n'
                        '    print(" (test 점수는 valid 점수와 조금 다를 수 있습니다)")\n'
                        '    print("=" * 60)\n'
                        '\n'
                        '\n'
                        'if __name__ == "__main__":\n'
                        '    main()\n'}

for name, source in SOURCES.items():
    path = WORK / name
    if path.exists() and path.read_text(encoding='utf-8') != source:
        raise RuntimeError(f'{path}에 다른 코드가 있습니다. 기존 파일을 확인하고 작업 폴더를 선택하세요.')
    path.write_text(source, encoding='utf-8')
print('준비 완료:', ', '.join(SOURCES))

## 데이터·평가 전처리·제출 규격 확인
모든 이미지의 경로와 RGB 크기를 확인합니다. 학습 라벨만으로 클래스 분포를 확인합니다.
개선 모델의 검증 전처리가 공식 검사기와 같은 텐서를 만드는지도 검사합니다.

In [ ]:
import csv
import importlib.util
from collections import Counter
from PIL import Image

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

data_module = load_module('leaf_dataset', WORK / 'dataset.py')
model_module = load_module('leaf_model', WORK / 'model.py')
checker_module = load_module('leaf_checker', WORK / 'check_submission.py')
assert not checker_module.check_imports(str(WORK / 'model.py'))
for split, expected in [('train', 8000), ('valid', 2284)]:
    dataset = data_module.PlantDataset(DATA_DIR, split, data_module.get_transforms(False))
    assert len(dataset) == expected, (split, len(dataset))
    for row in dataset.rows:
        with Image.open(dataset.img_dir / row['id']) as image:
            assert image.size == (128, 128) and image.mode == 'RGB', row['id']
    print(split, len(dataset), '장 확인')
    if split == 'train':
        print('학습 라벨 분포:', Counter((int(r['crop_label']), int(r['health_label'])) for r in dataset.rows))
sample = data_module.PlantDataset(DATA_DIR, 'valid', data_module.get_transforms(False))
with Image.open(sample.img_dir / sample.rows[0]['id']) as image:
    image = image.convert('RGB')
    assert torch.equal(data_module.get_transforms(False)(image), checker_module.EVAL_TRANSFORM(image))
model = model_module.MultiTaskCNN().eval()
params = sum(p.numel() for p in model.parameters())
buffers = sum(b.numel() for b in model.buffers() if b.is_floating_point())
assert params + buffers <= 2_000_000
with torch.no_grad():
    a, b = model(torch.randn(2, 3, 128, 128))
assert a.shape == (2, 3) and b.shape == (2, 2)
assert torch.isfinite(a).all() and torch.isfinite(b).all()
print(f'파라미터 {params:,} + 실수형 버퍼 {buffers:,} = {params + buffers:,}')
del model

## 실제 이미지로 학습 smoke check
소수 데이터로 1 epoch 학습하여 역전파·저장·평가가 연결되는지 확인합니다.
이 점수는 전체 검증 성능이 아니며 제출 파일로 사용하지 않습니다.

In [ ]:
import json
import tempfile

def run_logged(command, cwd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=str(cwd), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

smoke_dir = Path(tempfile.mkdtemp(prefix='leaf-smoke-', dir='/content'))
run_logged([sys.executable, '-u', 'train.py', '--data_dir', str(DATA_DIR),
            '--run_dir', str(smoke_dir), '--epochs', '1', '--batch_size', '8',
            '--train_limit', '32', '--valid_limit', '32', '--device', 'cuda'],
           WORK, smoke_dir / 'console.log')
assert json.loads((smoke_dir / 'best_metrics.json').read_text())['smoke_run']

## 공식 baseline 학습 (기본 설정 10 epoch)
완료한 baseline은 다시 학습하지 않습니다. 중단된 공식 baseline은 재개 기능이 없어
같은 폴더에 처음부터 다시 학습합니다. 개선 모델은 다음 셀에서 epoch 단위 재개를 지원합니다.

In [ ]:
import shutil

BASELINE_DIR = RUN_ROOT / f'baseline_official_{BASELINE_EPOCHS}epochs'
if RUN_BASELINE:
    BASELINE_DIR.mkdir(parents=True, exist_ok=True)
    for name in ['model.py', 'dataset.py', 'train.py']:
        source = OFFICIAL_PROJECT / name
        target = BASELINE_DIR / name
        if target.exists():
            assert target.read_bytes() == source.read_bytes(), f'Baseline 코드 변경: {target}'
        else:
            shutil.copy2(source, target)
    marker = BASELINE_DIR / 'completed.json'
    if not marker.exists():
        run_logged([sys.executable, '-u', str(OFFICIAL_PROJECT / 'train.py'),
                    '--data_dir', str(DATA_DIR), '--device', 'cuda',
                    '--epochs', str(BASELINE_EPOCHS), '--batch_size', '64', '--lr', '0.001',
                    '--save_path', str(BASELINE_DIR / 'best_model.pt')],
                   OFFICIAL_PROJECT, BASELINE_DIR / 'console.log')
        marker.write_text(json.dumps({'official_commit': OFFICIAL_COMMIT,
                                      'epochs': BASELINE_EPOCHS}), encoding='utf-8')
    else:
        assert json.loads(marker.read_text()) == {'official_commit': OFFICIAL_COMMIT, 'epochs': BASELINE_EPOCHS}
        assert (BASELINE_DIR / 'best_model.pt').is_file()
        print('완료한 baseline 재사용:', BASELINE_DIR)

## 개선 모델 전체 학습
각 epoch에서 raw와 EMA의 평균 Macro-F1을 비교해 최고 가중치를 저장합니다.
best_model.pt는 제출용 state_dict이고, last_checkpoint.pt는 optimizer 등을 포함한 재개 전용 파일입니다.
마지막 epoch까지 무조건 쓰지 않고 전체 학습 중 가장 좋은 검증 결과를 사용합니다.

In [ ]:
command = [sys.executable, '-u', 'train.py', '--data_dir', str(DATA_DIR),
           '--run_dir', str(RUN_DIR), '--epochs', str(EPOCHS), '--batch_size', str(BATCH_SIZE),
           '--lr', str(LEARNING_RATE), '--weight_decay', str(WEIGHT_DECAY),
           '--class_weight_power', str(CLASS_WEIGHT_POWER), '--seed', str(SEED), '--device', 'cuda']
if (RUN_DIR / 'last_checkpoint.pt').exists():
    command.append('--resume')
run_logged(command, WORK, RUN_DIR / 'console.log')
metrics = json.loads((RUN_DIR / 'best_metrics.json').read_text())
assert not metrics['smoke_run']
print('최적 모델:', json.dumps(metrics, ensure_ascii=False, indent=2))
history = json.loads((RUN_DIR / 'history.json').read_text())
print('epoch | loss | crop F1 | health F1 | 평균 F1 | 초')
for row in history:
    print(f"{row['epoch']:5d} | {row['train_loss']:.4f} | {row['crop_f1']:.4f} | "
          f"{row['health_f1']:.4f} | {row['score']:.4f} | {row['seconds']:.1f}")

## 공식 CPU 검사와 baseline 비교
전체 valid를 공식 전처리로 검사합니다. 두 학습 기간이 다르므로 이 비교는 전체 학습 방법의 비교입니다.
구조만의 효과를 확인하려면 두 모델을 동일 epoch·seed·학습 설정으로 따로 비교해야 합니다.
검사에 실패하면 제출 묶음을 만들지 않습니다.

In [ ]:
import re

candidates = [('improved', RUN_DIR)]
if RUN_BASELINE:
    candidates.append(('baseline', BASELINE_DIR))
results = []
for name, directory in candidates:
    checked = subprocess.run(
        [sys.executable, str(WORK / 'check_submission.py'), '--model', str(directory / 'model.py'),
         '--weights', str(directory / 'best_model.pt'), '--data_dir', str(DATA_DIR)],
        cwd=str(WORK), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    print(checked.stdout)
    (directory / 'submission_check.log').write_text(checked.stdout, encoding='utf-8')
    checked.check_returncode()
    match = re.search(r'평균 Macro-F1\s*:\s*([0-9.]+)', checked.stdout)
    assert match, '공식 점수를 찾지 못했습니다.'
    results.append({'name': name, 'score': float(match.group(1)), 'run_dir': str(directory)})
print('공식 검증 비교:', results)
selected = max(results, key=lambda row: row['score'])
print('검증 점수 기준 선택:', selected)
if RUN_BASELINE:
    lookup = {row['name']: row['score'] for row in results}
    print('개선 모델 - baseline:', round(lookup['improved'] - lookup['baseline'], 4))
(RUN_DIR / 'comparison.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

## 제출용 두 파일 묶기·다운로드
선택한 실험의 model.py와 best_model.pt만 묶습니다. checkpoint나 데이터는 제출하지 않습니다.
리더보드에는 zip을 풀어서 두 파일을 올리세요. 제출은 이 노트북이 자동으로 수행하지 않습니다.
다른 실험은 RUN_NAME을 바꾸고 학습·검사 셀을 실행하세요. 예: 가중치 해제(power=0),
가중치 강화(power=0.5), seed=43/44. 실제 검증 결과를 보고 설정을 선택하세요.

In [ ]:
import hashlib
import zipfile
from google.colab import files

selected_dir = Path(selected['run_dir'])
archive = RUN_DIR / 'submission.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in ['model.py', 'best_model.pt']:
        path = selected_dir / name
        if name == 'best_model.pt':
            assert path.stat().st_size <= 10 * 1024 * 1024
        bundle.write(path, arcname=name)
        print(name, path.stat().st_size, 'bytes', 'sha256:', hashlib.sha256(path.read_bytes()).hexdigest())
print('Drive 저장 완료:', archive)
files.download(str(archive))